In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockA (T) — 读取 T 数据 & 预处理 & 写缓存
#
# 功能：
#   1) 从 EXTR T.zm 文件中，计算 60–90°N 极区 minimum T(time, lev)
#   2) 从 merged 文件中，同样计算 60–90°N 极区 minimum T(time, lev)
#   3) 结果写入 NetCDF，供 BlockB / BlockC / BlockD 使用
#   4) O3 极端年信息从 General_O3 目录中读取（不再重新计算）
#
# 路径：
#   O3_ROOT = /home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3
#   T_ROOT  = /home/weiji/test_code/plots/New_weiji_general_why0008important/General_T
#
#   EXTR T 输入：
#     /mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/CO2x1SmidEmin_yBWCN.cam.h1.0101-0300_T.zm.nc
#   merged 输入：
#     /mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/BWCN.e122.f19_g16.merged.nc
#
#   输出：
#     T_min_EXTR_60_90N_lev_time.nc   — T_min_60_90N(time, lev) [K]
#     T_min_MERGED_60_90N_lev_time.nc — T_min_60_90N(time, lev) [K]
# ============================================================

import os
import numpy as np
import xarray as xr

O3_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
T_ROOT  = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_T"
os.makedirs(T_ROOT, exist_ok=True)

EXTR_T_PATH = (
    "/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/"
    "CO2x1SmidEmin_yBWCN.cam.h1.0101-0300_T.zm.nc"
)
MERGED_PATH = (
    "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
    "BWCN.e122.f19_g16.merged.nc"
)

LOW10_TXT = os.path.join(O3_ROOT, "O3_lowest10_years.txt")
LOW25_TXT = os.path.join(O3_ROOT, "O3_lowest25pct_years.txt")

T_EXTR_MIN_NC   = os.path.join(T_ROOT, "T_min_EXTR_60_90N_lev_time.nc")
T_MERGED_MIN_NC = os.path.join(T_ROOT, "T_min_MERGED_60_90N_lev_time.nc")


def calc_minT_pcap(ds, var="T"):
    """
    计算 60–90°N 极区 minimum T(time, lev):

      - 若有 lon → 先经向平均
      - 在纬度 60–90°N 区域上取最小值 (min over lat)
      - 垂直维统一命名为 'lev'，单位统一为 Pa

    返回 DataArray: T_min_60_90N(time, lev) [K]
    """
    da = ds[var]

    # zonal mean
    if "lon" in da.dims:
        da = da.mean(dim="lon")

    if "lat" not in da.dims:
        raise ValueError(f"{var} has no 'lat' dimension, dims={da.dims}")

    # 取 60–90N 极区 & 最低温度
    da_cap = da.sel(lat=slice(60, 90))
    da_min = da_cap.min(dim="lat")  # (time, vertical)

    # 识别垂直维
    lev_name = None
    for name in ("plev", "lev_p", "lev", "level"):
        if name in da_min.dims:
            lev_name = name
            break
    if lev_name is None:
        raise ValueError(
            f"{var}_min(60-90N) has no vertical dim among ('plev','lev_p','lev','level'), "
            f"dims={da_min.dims}"
        )

    lev_vals = da_min[lev_name].values
    max_lev = float(np.nanmax(lev_vals))

    # 统一为 Pa
    if max_lev <= 2000.0:
        lev_pa = lev_vals * 100.0
    else:
        lev_pa = lev_vals

    da_min = da_min.rename({lev_name: "lev"})
    da_min = da_min.assign_coords(lev=("lev", lev_pa))

    return da_min  # (time, lev)


def main_blockA_T():
    # ==== 检查 O3 极端年列表 ====
    if not (os.path.exists(LOW10_TXT) and os.path.exists(LOW25_TXT)):
        raise FileNotFoundError(
            "O3 extreme-year txt files not found. Please run O3 BlockA first.\n"
            f"Expected: {LOW10_TXT} and {LOW25_TXT}"
        )

    lowest10_years = np.loadtxt(LOW10_TXT, dtype=int).reshape(-1)
    lowest25_years = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)
    print("[BlockA_T] Lowest-10 O3 years:", lowest10_years)
    print("[BlockA_T] Lowest-25% O3 years:", lowest25_years)

    # ==== 1. EXTR T_min_60_90N(time, lev) ====
    print("[BlockA_T] Reading EXTR T.zm:", EXTR_T_PATH)
    ds_extr = xr.open_dataset(EXTR_T_PATH)
    t_extr_min = calc_minT_pcap(ds_extr, var="T")
    ds_extr.close()

    years_extr = np.unique(t_extr_min.time.dt.year.values.astype(int))
    n_days = int(t_extr_min.time.dt.dayofyear.max())
    print("[BlockA_T] EXTR T_min years:", years_extr[0], "→", years_extr[-1])
    print("[BlockA_T] EXTR n_years:", len(years_extr), "n_days:", n_days)

    t_extr_min = t_extr_min.transpose("time", "lev")
    t_extr_min.name = "T_min_60_90N"
    t_extr_min.attrs["long_name"] = "Minimum temperature over 60–90N"
    t_extr_min.attrs["units"] = "K"
    t_extr_min["lev"].attrs["units"] = "Pa"

    t_extr_min.to_netcdf(T_EXTR_MIN_NC)
    print("[BlockA_T] Saved EXTR T_min_60_90N(time,lev) to:", T_EXTR_MIN_NC)

    # ==== 2. merged T_min_60_90N(time, lev) ====
    print("[BlockA_T] Reading merged file:", MERGED_PATH)
    ds_merged = xr.open_dataset(MERGED_PATH)
    t_merged_min = calc_minT_pcap(ds_merged, var="T")
    ds_merged.close()

    years_merged = np.unique(t_merged_min.time.dt.year.values.astype(int))
    n_days_m = int(t_merged_min.time.dt.dayofyear.max())
    print("[BlockA_T] merged T_min years:", years_merged[0], "→", years_merged[-1])
    print("[BlockA_T] merged n_years:", len(years_merged), "n_days:", n_days_m)

    t_merged_min = t_merged_min.transpose("time", "lev")
    t_merged_min.name = "T_min_60_90N"
    t_merged_min.attrs["long_name"] = "Minimum temperature over 60–90N"
    t_merged_min.attrs["units"] = "K"
    t_merged_min["lev"].attrs["units"] = "Pa"

    t_merged_min.to_netcdf(T_MERGED_MIN_NC)
    print("[BlockA_T] Saved merged T_min_60_90N(time,lev) to:", T_MERGED_MIN_NC)

    print("[BlockA_T] Done.")


if __name__ == "__main__":
    main_blockA_T()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockB (T) — 60–90°N 极区最低温度折线图（10 / 50 / 100 hPa）
#
# 功能：
#   1) EXTR：10 个低 O3 年 vs 其它年份（T_min_60_90N at 10/50/100 hPa）
#   2) merged：0008/0013/0014/0019 vs EXTR all / low25 气候态
#      （T_min_60_90N at 10/50/100 hPa）
#
#   时间轴：前一年 Oct–Dec (91 天) + 当年 Jan–Sep，x=0..364
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
from matplotlib import cm
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib import rcParams

O3_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
T_ROOT  = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_T"
os.makedirs(T_ROOT, exist_ok=True)

T_EXTR_MIN_NC   = os.path.join(T_ROOT, "T_min_EXTR_60_90N_lev_time.nc")
T_MERGED_MIN_NC = os.path.join(T_ROOT, "T_min_MERGED_60_90N_lev_time.nc")
LOW10_TXT       = os.path.join(O3_ROOT, "O3_lowest10_years.txt")
LOW25_TXT       = os.path.join(O3_ROOT, "O3_lowest25pct_years.txt")

N_PREV = 91
YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)


def get_level_index(lev_vals_pa, target_hpa):
    target_pa = target_hpa * 100.0
    idx = int(np.argmin(np.abs(lev_vals_pa - target_pa)))
    return idx, float(lev_vals_pa[idx] / 100.0)


def plot_extremes_vs_others(var_extr, lowest10_years, level_label, out_png, out_pdf):
    years_extr = np.unique(var_extr.time.dt.year.values.astype(int))
    n_days = int(var_extr.time.dt.dayofyear.max())
    print(f"[BlockB_T] EXTR years for {level_label}:", years_extr)
    print(f"[BlockB_T] Days per year (EXTR): {n_days}")

    fig, ax = plt.subplots(1, 1, figsize=(13, 10), constrained_layout=True)

    low_years = lowest10_years
    neutral_years = years_extr[~np.isin(years_extr, low_years)]

    var_low_cur = var_extr.sel(time=var_extr.time.dt.year.isin(low_years))
    var_low_prev = var_extr.sel(time=var_extr.time.dt.year.isin(low_years - 1))

    var_neu_cur = var_extr.sel(time=var_extr.time.dt.year.isin(neutral_years))
    var_neu_prev = var_extr.sel(time=var_extr.time.dt.year.isin(neutral_years - 1))

    neu_cur_mean = var_neu_cur.groupby("time.dayofyear").mean("time")
    neu_cur_std  = var_neu_cur.groupby("time.dayofyear").std("time")
    neu_prev_mean = var_neu_prev.groupby("time.dayofyear").mean("time")
    neu_prev_std  = var_neu_prev.groupby("time.dayofyear").std("time")

    neu_cur_mean = np.array(neu_cur_mean)
    neu_cur_std  = np.array(neu_cur_std)
    neu_prev_mean = np.array(neu_prev_mean)
    neu_prev_std  = np.array(neu_prev_std)

    n_extreme = len(low_years)
    var_low_cur_arr  = np.reshape(np.array(var_low_cur), (n_extreme, n_days))
    var_low_prev_arr = np.reshape(np.array(var_low_prev), (n_extreme, n_days))

    colors_ext = cm.twilight(np.linspace(0, 1, n_extreme))

    for j in range(n_extreme):
        ax.plot(
            range(N_PREV, n_days),
            var_low_cur_arr[j, 0:n_days - N_PREV],
            color=colors_ext[j],
            alpha=0.8,
            linewidth=2,
            label="low O3 years" if j == 0 else None,
        )
        ax.plot(
            range(0, N_PREV),
            var_low_prev_arr[j, n_days - N_PREV:n_days],
            color=colors_ext[j],
            alpha=0.8,
            linewidth=2,
        )

    ax.errorbar(
        range(N_PREV, n_days),
        neu_cur_mean[0:n_days - N_PREV],
        neu_cur_std[0:n_days - N_PREV],
        color="grey",
        alpha=0.5,
        elinewidth=1.5,
        linewidth=3,
        label="all other years",
    )
    ax.errorbar(
        range(0, N_PREV),
        neu_prev_mean[n_days - N_PREV:n_days],
        neu_prev_std[n_days - N_PREV:n_days],
        color="grey",
        alpha=0.5,
        elinewidth=1.5,
        linewidth=3,
    )

    ax.set_xlim(0, n_days)
    ax.set_xticks([0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334])
    ax.set_xticklabels(
        ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr",
         "May", "June", "July", "Aug", "Sep"],
        fontsize=15,
    )

    ax.set_ylabel(f"Min T over 60–90°N, {level_label} (K)", fontsize=18)
    ax.set_yticklabels(ax.get_yticks(), size=18)
    ax.legend(fontsize=14)
    ax.set_title(f"Polar-cap minimum T (60–90°N), {level_label} — low O$_3$ years vs others")
    ax.axhline(y=197, color='royalblue', linestyle='--', linewidth=1.5)
    ax.text(
        5, 194,
        'Cl activation threshold',
        horizontalalignment='left',
        fontsize=20,
        color='royalblue'
    )

    plt.savefig(out_png, dpi=300)
    plt.savefig(out_pdf)
    plt.close(fig)

    print("[BlockB_T] Saved extreme vs other years figure:")
    print("   ", out_png)
    print("   ", out_pdf)


def plot_special_vs_climatology(
    var_extr,
    var_merged,
    lowest25_years,
    level_label,
    out_png,
    out_pdf,
):
    years_extr = np.unique(var_extr.time.dt.year.values.astype(int))
    n_days = int(var_extr.time.dt.dayofyear.max())
    print(f"[BlockB_T] EXTR years ({level_label}):", years_extr)
    print(f"[BlockB_T] Days per year:", n_days)

    # ===== EXTR all-year 日气候态（保持不变） =====
    clim_all_mean = var_extr.groupby("time.dayofyear").mean("time")
    clim_all_std  = var_extr.groupby("time.dayofyear").std("time")

    # ===== low-25% 气候态：Y-1 Oct–Dec + Y Jan–Sep =====
    base_low25_cur = var_extr.sel(time=var_extr.time.dt.year.isin(lowest25_years))
    base_low25_prev = var_extr.sel(time=var_extr.time.dt.year.isin(lowest25_years - 1))

    clim_low25_cur_mean = base_low25_cur.groupby("time.dayofyear").mean("time")
    clim_low25_cur_std  = base_low25_cur.groupby("time.dayofyear").std("time")
    clim_low25_prev_mean = base_low25_prev.groupby("time.dayofyear").mean("time")
    clim_low25_prev_std  = base_low25_prev.groupby("time.dayofyear").std("time")

    clim_all_mean = np.array(clim_all_mean)
    clim_all_std  = np.array(clim_all_std)

    clim_low25_cur_mean = np.array(clim_low25_cur_mean)
    clim_low25_cur_std  = np.array(clim_low25_cur_std)
    clim_low25_prev_mean = np.array(clim_low25_prev_mean)
    clim_low25_prev_std  = np.array(clim_low25_prev_std)

    all_prev_mean = clim_all_mean[n_days - N_PREV:n_days]
    all_prev_std  = clim_all_std[n_days - N_PREV:n_days]
    all_cur_mean  = clim_all_mean[0:n_days - N_PREV]
    all_cur_std   = clim_all_std[0:n_days - N_PREV]

    all_comp_mean = np.concatenate([all_prev_mean, all_cur_mean])
    all_comp_std  = np.concatenate([all_prev_std,  all_cur_std])

    low25_prev_mean = clim_low25_prev_mean[n_days - N_PREV:n_days]
    low25_prev_std  = clim_low25_prev_std[n_days - N_PREV:n_days]
    low25_cur_mean  = clim_low25_cur_mean[0:n_days - N_PREV]
    low25_cur_std   = clim_low25_cur_std[0:n_days - N_PREV]

    low25_comp_mean = np.concatenate([low25_prev_mean, low25_cur_mean])
    low25_comp_std  = np.concatenate([low25_prev_std,  low25_cur_std])

    years_merged = np.unique(var_merged.time.dt.year.values.astype(int))
    print(f"[BlockB_T] merged years ({level_label}):", years_merged)

    missing_special = YEARS_SPECIAL[~np.isin(YEARS_SPECIAL, years_merged)]
    if len(missing_special) > 0:
        print(f"⚠️ Warning: these special years are missing in merged ({level_label}):",
              missing_special)

    rcParams.update({
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans"],
        "font.size": 9,
        "axes.titlesize": 10,
        "axes.labelsize": 10,
        "axes.linewidth": 0.8,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "axes.spines.top": False,
        "axes.spines.right": False,
    })

    fig, ax = plt.subplots(1, 1, figsize=(6.5, 4.0), constrained_layout=True)

    x_full = np.arange(n_days)
    colors_spec = plt.cm.tab10(np.linspace(0, 1, len(YEARS_SPECIAL)))
    handles_years = []

    for i, y in enumerate(YEARS_SPECIAL):
        if y not in years_merged:
            continue

        cur = var_merged.sel(time=var_merged.time.dt.year == y)
        prev = var_merged.sel(time=var_merged.time.dt.year == (y - 1))

        if (cur.size < n_days) or (prev.size < n_days):
            print(f"⚠️ Year {y:04d} or {y-1:04d} does not have {n_days} days ({level_label}), skip.")
            continue

        cur_arr  = np.array(cur)
        prev_arr = np.array(prev)

        series_comp = np.concatenate([
            prev_arr[n_days - N_PREV:n_days],
            cur_arr[0:n_days - N_PREV],
        ])

        ax.plot(
            x_full,
            series_comp,
            linestyle="-",
            linewidth=1.5,
            color=colors_spec[i],
            label=f"Year {int(y):04d}",
        )
        handles_years.append(
            Line2D([0], [0], color=colors_spec[i], lw=1.5, label=f"Year {int(y):04d}")
        )

    ax.fill_between(
        x_full,
        all_comp_mean - all_comp_std,
        all_comp_mean + all_comp_std,
        color="0.85",
        alpha=0.9,
        linewidth=0,
        zorder=0,
    )
    ax.plot(
        x_full,
        all_comp_mean,
        linestyle="--",
        linewidth=1.8,
        color="black",
        label="EXTR climatology (all years)",
    )

    ax.fill_between(
        x_full,
        low25_comp_mean - low25_comp_std,
        low25_comp_mean + low25_comp_std,
        facecolor="none",
        edgecolor="0.5",
        hatch="///",
        linewidth=0.0,
        zorder=1,
    )
    ax.plot(
        x_full,
        low25_comp_mean,
        linestyle="-",
        linewidth=2.0,
        color="black",
        label="EXTR low-ozone composite",
    )

    ax.set_xlim(0, n_days)
    ax.set_xticks([0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334])
    ax.set_xticklabels(
        ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr",
         "May", "June", "July", "Aug", "Sep"]
    )
    ax.set_ylabel(f"Min T over 60–90°N, {level_label} (K)")

    patch_all = Patch(facecolor="0.85", edgecolor="none", label="EXTR all-year ±1σ")
    patch_low = Patch(facecolor="none", edgecolor="0.5", hatch="///",
                      label="EXTR low-ozone ±1σ")

    line_all = Line2D([0], [0], color="black", lw=1.8, ls="--",
                      label="EXTR climatology (all years)")
    line_low = Line2D([0], [0], color="black", lw=2.0, ls="-",
                      label="EXTR low-ozone composite")

    handles = handles_years + [line_all, patch_all, line_low, patch_low]
    ax.legend(handles=handles, loc="best", frameon=False, fontsize=8, ncol=2)

    ax.set_title(f"Polar-cap minimum T (60–90°N), {level_label}")
    ax.grid(False)

    ax.axhline(y=197, color='royalblue', linestyle='--', linewidth=1.2)
    ax.text(
        5, 194,
        'Cl activation threshold',
        horizontalalignment='left',
        fontsize=12,
        color='royalblue'
    )

    plt.savefig(out_png, dpi=300)
    plt.savefig(out_pdf)
    plt.close(fig)

    print("[BlockB_T] Saved special-years vs climatology figure:")
    print("   ", out_png)
    print("   ", out_pdf)


def main_blockB_T():
    t_extr_min = xr.load_dataarray(T_EXTR_MIN_NC)    # (time, lev[Pa])
    t_merged_min = xr.load_dataarray(T_MERGED_MIN_NC)

    lev_vals = t_extr_min["lev"].values

    lowest10_years = np.loadtxt(LOW10_TXT, dtype=int).reshape(-1)
    lowest25_years = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)
    print("[BlockB_T] Lowest 10 O3 years:", lowest10_years)
    print("[BlockB_T] Lowest 25% O3 years:", lowest25_years)

    # ===== [MOD] add 10 hPa =====
    target_levels_hpa = [10.0, 50.0, 100.0]

    for target_hpa in target_levels_hpa:
        idx, lev_actual_hpa = get_level_index(lev_vals, target_hpa)
        level_label = f"{lev_actual_hpa:.1f} hPa"

        var_extr = t_extr_min.isel(lev=idx)
        var_merged = t_merged_min.isel(lev=idx)

        print(f"\n[BlockB_T] Processing level ~{target_hpa} hPa (index {idx}), "
              f"actual {lev_actual_hpa:.2f} hPa")

        out_png1 = os.path.join(
            T_ROOT, f"Tmin_60_90N_{int(round(lev_actual_hpa))}hPa_evolution_min_10extr_200yrs.png"
        )
        out_pdf1 = os.path.join(
            T_ROOT, f"Tmin_60_90N_{int(round(lev_actual_hpa))}hPa_evolution_min_10extr_200yrs.pdf"
        )
        plot_extremes_vs_others(
            var_extr=var_extr,
            lowest10_years=lowest10_years,
            level_label=level_label,
            out_png=out_png1,
            out_pdf=out_pdf1,
        )

        out_png2 = os.path.join(
            T_ROOT,
            f"Tmin_60_90N_{int(round(lev_actual_hpa))}hPa_daily_special_years_vs_extr_climatology.png"
        )
        out_pdf2 = os.path.join(
            T_ROOT,
            f"Tmin_60_90N_{int(round(lev_actual_hpa))}hPa_daily_special_years_vs_extr_climatology.pdf"
        )
        plot_special_vs_climatology(
            var_extr=var_extr,
            var_merged=var_merged,
            lowest25_years=lowest25_years,
            level_label=level_label,
            out_png=out_png2,
            out_pdf=out_pdf2,
        )

    print("[BlockB_T] Done.")


if __name__ == "__main__":
    main_blockB_T()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockC (T) — 60–90°N minimum T 垂直时间剖面 anomaly + 显著性
#
# 输出 NetCDF:
#   T_min_vert_anom_special_60_90N.nc
#
# coords:
#   ref_year (n_ref) : [8, 13, 14, 19]
#   lev             : 垂直层 (Pa)
#   time_index      : 0..364 （拼接后的时间）
#   dayofyear       : [275..365, 1..274]
#
# variables (K):
#   anom_all        (ref_year, lev, time_index)
#   anom_low25      (ref_year, lev, time_index)
#   clim_all_comp   (lev, time_index)
#   clim_low25_comp (lev, time_index)
#
# significance mask (True = 显著):
#   t_sig_all, b_sig_all, t_sig_low25, b_sig_low25
#
# 重要修正：
#   low-25% 气候态采用 “前一年 Oct–Dec + 当年 Jan–Sep”的定义，
#   与线图的时间拼接一致，避免 1.1 与 12.31 之间的假跳跃。
# ============================================================

import os
import numpy as np
import xarray as xr
from scipy.stats import t as t_dist

O3_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_O3"
T_ROOT  = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_T"
os.makedirs(T_ROOT, exist_ok=True)

EXTR_T_PATH = (
    "/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/"
    "CO2x1SmidEmin_yBWCN.cam.h1.0101-0300_T.zm.nc"
)
MERGED_FILE = (
    "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
    "BWCN.e122.f19_g16.merged.nc"
)

LOW25_TXT = os.path.join(O3_ROOT, "O3_lowest25pct_years.txt")

OUT_NC = os.path.join(T_ROOT, "T_min_vert_anom_special_60_90N.nc")

N_PREV = 91
YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)


def calc_minT_pcap(ds, var="T"):
    """
    计算 60–90°N 极区 minimum T(time, lev)，lev 统一为 Pa。
    """
    da = ds[var]
    if "lon" in da.dims:
        da = da.mean(dim="lon")

    if "lat" not in da.dims:
        raise ValueError(f"{var} has no 'lat' dimension, dims={da.dims}")

    da_cap = da.sel(lat=slice(60, 90))
    da_min = da_cap.min(dim="lat")  # (time, vertical)

    lev_name = None
    for name in ("plev", "lev_p", "lev", "level"):
        if name in da_min.dims:
            lev_name = name
            break
    if lev_name is None:
        raise ValueError(
            f"{var}_min(60-90N) has no vertical dim among ('plev','lev_p','lev','level'), "
            f"dims={da_min.dims}"
        )

    lev_vals = da_min[lev_name].values
    max_lev = float(np.nanmax(lev_vals))

    if max_lev <= 2000.0:
        lev_pa = lev_vals * 100.0
    else:
        lev_pa = lev_vals

    da_min = da_min.rename({lev_name: "lev"})
    da_min = da_min.assign_coords(lev=("lev", lev_pa))

    return da_min  # (time, lev), K


def bootstrap_ci(data, nboot=5000, alpha=0.05, rng=None):
    data = np.asarray(data)
    data = data[~np.isnan(data)]
    n = data.size
    if n < 2:
        return np.nan, np.nan

    if rng is None:
        rng = np.random.default_rng()

    means = np.empty(nboot)
    for i in range(nboot):
        samp = rng.choice(data, size=n, replace=True)
        means[i] = np.mean(samp)

    low = np.percentile(means, 100 * alpha / 2.0)
    high = np.percentile(means, 100 * (1 - alpha / 2.0))
    return low, high


def compute_significance_for_baseline(
    base_anom,
    anom_ref,
    doy_base,
    doy_comp,
    alpha=0.05,
    nboot=5000,
):
    base_vals = base_anom.values  # (time, lev)
    lev_n = base_vals.shape[1]
    t_len = anom_ref.shape[1]

    t_sig = np.zeros((lev_n, t_len), dtype=bool)
    b_sig = np.zeros((lev_n, t_len), dtype=bool)

    rng = np.random.default_rng(2025)

    for ti in range(t_len):
        day = int(doy_comp[ti])
        mask_day = (doy_base == day)
        if not np.any(mask_day):
            continue

        day_samples = base_vals[mask_day, :]  # (n_samples, lev)

        for li in range(lev_n):
            col = day_samples[:, li]
            col = col[~np.isnan(col)]
            if col.size < 2:
                continue

            obs = anom_ref[li, ti]
            se = np.std(col, ddof=1) / np.sqrt(col.size)
            if se == 0:
                continue
            tstat = obs / se
            pval = 2 * (1 - t_dist.cdf(abs(tstat), df=col.size - 1))
            t_sig[li, ti] = (pval < alpha)

            lo, hi = bootstrap_ci(col, nboot=nboot, alpha=alpha, rng=rng)
            if np.isnan(lo) or np.isnan(hi):
                continue
            b_sig[li, ti] = not (lo <= obs <= hi)

    return t_sig, b_sig


def main_blockC_T():
    # ===== 读取 O3 low25 年份 =====
    lowest25_years = np.loadtxt(LOW25_TXT, dtype=int).reshape(-1)
    print("[BlockC_T] Lowest 25% O3 years:", lowest25_years)

    # ===== 1. EXTR T_min_60_90N(time,lev) =====
    print("[BlockC_T] Reading EXTR T.zm:", EXTR_T_PATH)
    ds_extr = xr.open_dataset(EXTR_T_PATH)
    t_extr_min = calc_minT_pcap(ds_extr, "T")
    ds_extr.close()

    doy_extr = t_extr_min.time.dt.dayofyear.values.astype(int)
    years_extr = t_extr_min.time.dt.year.values.astype(int)
    print("[BlockC_T] EXTR years:", np.unique(years_extr))

    n_days = int(t_extr_min.time.dt.dayofyear.max())
    print("[BlockC_T] Days per year (EXTR):", n_days)

    # ===== 2. EXTR 日气候态（全体 & low25） =====
    # 全部年份气候态
    clim_all = t_extr_min.groupby("time.dayofyear").mean("time")

    # === [FIX] low-25% 气候态：Y-1 Oct–Dec + Y Jan–Sep ===
    base_low25_cur = t_extr_min.sel(time=t_extr_min.time.dt.year.isin(lowest25_years))
    base_low25_prev = t_extr_min.sel(
        time=t_extr_min.time.dt.year.isin(lowest25_years - 1)
    )

    clim_low25_cur = base_low25_cur.groupby("time.dayofyear").mean("time")
    clim_low25_prev = base_low25_prev.groupby("time.dayofyear").mean("time")

    # 构造一个完整的 low25 气候态 (1..365)：先拷贝“当年”的，再覆盖 Oct–Dec 段
    clim_low25_full = clim_low25_cur.copy()
    oct_dec = np.arange(n_days - N_PREV + 1, n_days + 1, dtype=int)
    clim_low25_full.loc[dict(dayofyear=oct_dec)] = (
        clim_low25_prev.sel(dayofyear=oct_dec).values
    )

    doys = clim_all["dayofyear"].values.astype(int)

    # ===== 3. merged T_min_60_90N(time,lev) =====
    print("[BlockC_T] Reading merged file:", MERGED_FILE)
    ds_merged = xr.open_dataset(MERGED_FILE)
    t_merged_min = calc_minT_pcap(ds_merged, "T")
    ds_merged.close()

    years_merged = t_merged_min.time.dt.year.values.astype(int)
    print("[BlockC_T] merged years:", np.unique(years_merged))

    lev = t_merged_min["lev"].values
    lev_n = lev.size

    # ===== 4. 拼接 DOY 序列 =====
    doy_comp = np.concatenate([
        np.arange(n_days - N_PREV + 1, n_days + 1, dtype=int),
        np.arange(1, n_days - N_PREV + 1, dtype=int),
    ])
    t_len = doy_comp.size
    print("[BlockC_T] Composite DOY sequence:", doy_comp[:5], "...", doy_comp[-5:])

    # ===== 5. baseline anomalies =====
    clim_all_sel_for_extr = clim_all.sel(dayofyear=t_extr_min.time.dt.dayofyear)
    base_anom_all = t_extr_min - clim_all_sel_for_extr

    # [FIX] low-25% baseline 使用 clim_low25_full
    clim_low25_sel_for_extr = clim_low25_full.sel(
        dayofyear=t_extr_min.time.dt.dayofyear
    )
    base_anom_low25 = t_extr_min - clim_low25_sel_for_extr

    # ===== 6. baseline climatology (拼接) =====
    clim_all_comp = clim_all.sel(dayofyear=doy_comp)
    # [FIX] 用 clim_low25_full，而不是原来的 clf_low25
    clim_low25_comp = clim_low25_full.sel(dayofyear=doy_comp)

    clim_all_comp_vals = clim_all_comp.transpose("lev", "dayofyear").values
    clim_low25_comp_vals = clim_low25_comp.transpose("lev", "dayofyear").values

    # ===== 7. special years anomalies & significance =====
    n_ref = len(YEARS_SPECIAL)

    anom_all_arr = np.zeros((n_ref, lev_n, t_len))
    anom_low_arr = np.zeros((n_ref, lev_n, t_len))

    t_sig_all = np.zeros((n_ref, lev_n, t_len), dtype=bool)
    b_sig_all = np.zeros((n_ref, lev_n, t_len), dtype=bool)
    t_sig_low = np.zeros((n_ref, lev_n, t_len), dtype=bool)
    b_sig_low = np.zeros((n_ref, lev_n, t_len), dtype=bool)

    for yi, y in enumerate(YEARS_SPECIAL):
        if y not in years_merged:
            print(f"[BlockC_T] ⚠️ Year {y:04d} not found in merged, skip.")
            continue

        ref_cur = t_merged_min.sel(time=t_merged_min.time.dt.year == y)
        ref_prev = t_merged_min.sel(time=t_merged_min.time.dt.year == (y - 1))

        if ref_cur.time.size < n_days or ref_prev.time.size < n_days:
            print(f"[BlockC_T] ⚠️ Year {y:04d} or {y-1:04d} does not have {n_days} days, skip.")
            continue

        ref_cur_vals = np.array(ref_cur.transpose("time", "lev").values)
        ref_prev_vals = np.array(ref_prev.transpose("time", "lev").values)

        ref_comp_vals = np.concatenate([
            ref_prev_vals[n_days - N_PREV:n_days, :],
            ref_cur_vals[0:n_days - N_PREV, :],
        ], axis=0)  # (t_len, lev_n)

        ref_comp = ref_comp_vals.T  # (lev_n, t_len)

        anom_all = ref_comp - clim_all_comp_vals
        anom_low = ref_comp - clim_low25_comp_vals

        anom_all_arr[yi, :, :] = anom_all
        anom_low_arr[yi, :, :] = anom_low

        print(f"[BlockC_T] Computing significance for year {y:04d} vs ALL baseline ...")
        t_all, b_all = compute_significance_for_baseline(
            base_anom_all,
            anom_all,
            doy_extr,
            doy_comp,
            alpha=0.05,
            nboot=5000,
        )
        print(f"[BlockC_T] Computing significance for year {y:04d} vs LOW25 baseline ...")
        t_low, b_low = compute_significance_for_baseline(
            base_anom_low25,
            anom_low,
            doy_extr,
            doy_comp,
            alpha=0.05,
            nboot=5000,
        )

        t_sig_all[yi, :, :] = t_all
        b_sig_all[yi, :, :] = b_all
        t_sig_low[yi, :, :] = t_low
        b_sig_low[yi, :, :] = b_low

    # ===== 8. 保存到 NetCDF =====
    ds_out = xr.Dataset(
        coords={
            "ref_year": ("ref_year", YEARS_SPECIAL),
            "lev": ("lev", lev),
            "time_index": ("time_index", np.arange(t_len)),
            "dayofyear": ("time_index", doy_comp),
        },
        data_vars={
            "anom_all": (("ref_year", "lev", "time_index"), anom_all_arr),
            "anom_low25": (("ref_year", "lev", "time_index"), anom_low_arr),
            "clim_all_comp": (("lev", "time_index"), clim_all_comp_vals),
            "clim_low25_comp": (("lev", "time_index"), clim_low25_comp_vals),
            "t_sig_all": (("ref_year", "lev", "time_index"), t_sig_all),
            "b_sig_all": (("ref_year", "lev", "time_index"), b_sig_all),
            "t_sig_low25": (("ref_year", "lev", "time_index"), t_sig_low),
            "b_sig_low25": (("ref_year", "lev", "time_index"), b_sig_low),
        },
    )

    ds_out["anom_all"].attrs["units"] = "K"
    ds_out["anom_low25"].attrs["units"] = "K"
    ds_out["clim_all_comp"].attrs["units"] = "K"
    ds_out["clim_low25_comp"].attrs["units"] = "K"
    ds_out["lev"].attrs["units"] = "Pa"

    ds_out.attrs["description"] = (
        "Minimum temperature over 60–90N, time–height anomalies (K) "
        "for special years relative to EXTR all-years and low-25% climatology. "
        "Low-25% climatology is constructed using previous-year Oct–Dec and "
        "current-year Jan–Sep for the lowest-25% years. "
        "Significance masks (t-test & bootstrap) included."
    )

    ds_out.to_netcdf(OUT_NC)
    print(f"[BlockC_T] Saved vertical anomaly dataset to: {OUT_NC}")
    print("[BlockC_T] Done.")


if __name__ == "__main__":
    main_blockC_T()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockD (T) — 绘制 60–90°N 最低温度时间×高度 anomaly 图
#
# 使用 BlockC_T 输出：
#   T_min_vert_anom_special_60_90N.nc
#
# 对每个 ref_year（8, 13, 14, 19）：
#   - baseline = all-years: t-test & bootstrap 的非显著区
#   - baseline = low-25%:  t-test & bootstrap 的非显著区
#
# 横坐标：0..364，与其它 block 一致，刻度 Oct–Sep
# 纵坐标：pressure (log scale)
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

T_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_T"
os.makedirs(T_ROOT, exist_ok=True)

IN_NC = os.path.join(T_ROOT, "T_min_vert_anom_special_60_90N.nc")

XTICKS = [0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334]
XTICK_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr",
                "May", "June", "July", "Aug", "Sep"]

mpl.rc("xtick", direction="out", labelsize=8)
mpl.rc("ytick", direction="out", labelsize=8)
mpl.rcParams["xtick.major.width"] = 0.8
mpl.rcParams["ytick.major.width"] = 0.8
mpl.rc("font", family="sans-serif")
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Liberation Sans"]
mpl.rc("axes", titlesize=11, labelsize=9, linewidth=0.8)
mpl.rc("legend", fontsize=7, frameon=False)
mpl.rc("figure", figsize=(6, 4), dpi=300)
mpl.rc("savefig", bbox="tight", pad_inches=0.1)


def guess_p_2_pa(lev_vals):
    lev_vals = np.asarray(lev_vals)
    maxv = np.nanmax(lev_vals)
    if maxv <= 2000.0:
        return lev_vals * 100.0, "hPa"
    else:
        return lev_vals, "Pa"


def plot_time_height_anom(
    x_idx,
    lev_vals,
    anom_vals,
    clim_vals,
    nonsig_mask,
    ref_year,
    baseline_label,
    test_label,
    outfile,
):
    fig, ax = plt.subplots()

    x = np.asarray(x_idx)
    p_pa, p_unit = guess_p_2_pa(lev_vals)

    vlim = np.nanmax(np.abs(anom_vals))
    if np.isfinite(vlim) and vlim > 0:
        vmax = min(vlim, 20.0)  # 温度 anomaly，一般 20K 够用了
    else:
        vmax = 10.0
    levels = np.linspace(-vmax, vmax, 31)

    cf = ax.contourf(
        x, p_pa, anom_vals,
        levels=levels,
        cmap="RdBu_r",
        extend="both",
        antialiased=True,
        alpha=0.85,
    )

    clim_mean = np.nanmean(clim_vals)
    clim_std = np.nanstd(clim_vals)
    if np.isfinite(clim_std) and clim_std > 0:
        clim_levels = np.linspace(clim_mean - 1.5 * clim_std,
                                  clim_mean + 1.5 * clim_std, 7)
    else:
        clim_levels = 7

    CS = ax.contour(
        x, p_pa, clim_vals,
        levels=clim_levels,
        colors="k",
        linewidths=0.7,
    )
    ax.clabel(CS, inline=True, fontsize=6, fmt="%.1f")

    if nonsig_mask is not None:
        mask_int = nonsig_mask.astype(int)
        ax.contourf(
            x, p_pa, mask_int,
            levels=[0.5, 1.5],
            colors="none",
            hatches=["///"],
            alpha=0,
        )
        patch = Patch(
            facecolor="white",
            hatch="///",
            edgecolor="black",
            label="Not significant (p ≥ 0.05)",
        )
        ax.legend(handles=[patch], loc="upper right")

    ax.set_yscale("log")
    ax.invert_yaxis()
    ax.set_ylabel(f"Pressure ({p_unit})")

    ax.set_xlim(x[0], x[-1])
    ax.set_xticks(XTICKS)
    ax.set_xticklabels(XTICK_LABELS)

    ax.grid(True, which="major", linestyle="--", linewidth=0.4, alpha=0.5)

    ax.set_title(
        f"Min T anomaly (K), 60–90°N, Year {int(ref_year):04d}\n"
        f"Baseline: {baseline_label}, Mask: non-significant by {test_label}"
    )

    cbar = plt.colorbar(cf, ax=ax, pad=0.02)
    cbar.set_label("Min T anomaly (K)")

    plt.savefig(outfile, dpi=300)
    plt.close(fig)
    print("  Saved:", outfile)


def main_blockD_T():
    ds = xr.open_dataset(IN_NC)

    ref_years = ds["ref_year"].values
    lev = ds["lev"].values
    time_index = ds["time_index"].values

    anom_all = ds["anom_all"].values
    anom_low = ds["anom_low25"].values
    clim_all = ds["clim_all_comp"].values
    clim_low = ds["clim_low25_comp"].values

    t_sig_all = ds["t_sig_all"].values
    b_sig_all = ds["b_sig_all"].values
    t_sig_low = ds["t_sig_low25"].values
    b_sig_low = ds["b_sig_low25"].values

    ds.close()

    for yi, y in enumerate(ref_years):
        # baseline all-years, t-test
        nonsig_t_all = ~t_sig_all[yi, :, :]
        outfile = os.path.join(
            T_ROOT,
            f"Tmin_anom_60_90N_year{int(y):04d}_vsALL_t_nonsig.png"
        )
        plot_time_height_anom(
            time_index,
            lev,
            anom_all[yi, :, :],
            clim_all,
            nonsig_t_all,
            ref_year=y,
            baseline_label="EXTR all-years climatology",
            test_label="t-test",
            outfile=outfile,
        )

        # baseline all-years, bootstrap
        nonsig_b_all = ~b_sig_all[yi, :, :]
        outfile = os.path.join(
            T_ROOT,
            f"Tmin_anom_60_90N_year{int(y):04d}_vsALL_bootstrap_nonsig.png"
        )
        plot_time_height_anom(
            time_index,
            lev,
            anom_all[yi, :, :],
            clim_all,
            nonsig_b_all,
            ref_year=y,
            baseline_label="EXTR all-years climatology",
            test_label="bootstrap",
            outfile=outfile,
        )

        # baseline low-25, t-test
        nonsig_t_low = ~t_sig_low[yi, :, :]
        outfile = os.path.join(
            T_ROOT,
            f"Tmin_anom_60_90N_year{int(y):04d}_vsLOW25_t_nonsig.png"
        )
        plot_time_height_anom(
            time_index,
            lev,
            anom_low[yi, :, :],
            clim_low,
            nonsig_t_low,
            ref_year=y,
            baseline_label="EXTR low-25% climatology",
            test_label="t-test",
            outfile=outfile,
        )

        # baseline low-25, bootstrap
        nonsig_b_low = ~b_sig_low[yi, :, :]
        outfile = os.path.join(
            T_ROOT,
            f"Tmin_anom_60_90N_year{int(y):04d}_vsLOW25_bootstrap_nonsig.png"
        )
        plot_time_height_anom(
            time_index,
            lev,
            anom_low[yi, :, :],
            clim_low,
            nonsig_b_low,
            ref_year=y,
            baseline_label="EXTR low-25% climatology",
            test_label="bootstrap",
            outfile=outfile,
        )

    print("[BlockD_T] All figures generated.")


if __name__ == "__main__":
    main_blockD_T()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockG (EHF, ALL 15d-smoothed)
# — 45–75°N v'T' 垂直时间剖面 std.anom + 显著性
#
# 核心设定（对比旧版）：
#   1) EXTR HFLUX(time, lev) 先在 time 上做 15-day centered running mean，
#      得到 ehf_extr_smooth。
#   2) daily climatology (mean/std) 基于 ehf_extr_smooth 计算：
#         clim_all(day, lev), clim_std(day, lev)
#   3) baseline anomalies: ehf_extr_smooth - clim_all(doy, lev)
#   4) INT v'T'（经向涡动热通量）在 45–75°N + 插值到 lev_keep 后，
#      也在 time 上做 15-day centered running mean。
#   5) composite 构造仍为 [Y-1 Oct–Dec, Y Jan–Sep] → t=0..364。
#   6) anomaly/std.anom 均相对于 “15d-smoothed EXTR daily climatology” 计算。
#
# 输出：
#   EHF_vert_std_anom_15dALL_special_45_75N.nc
# ============================================================

import os
import numpy as np
import xarray as xr
from scipy.stats import t as t_dist

EHF_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_EHF"
os.makedirs(EHF_ROOT, exist_ok=True)

EXTR_EHF_FILE = (
    "/mnt/backup_ETH/EXTR_2000/CO2x1SmidEmin_yBWCN/"
    "CO2x1SmidEmin_yBWCN.cam.h1.0100-0299_T2Mz_ehflux.isobar.nc"
)

INT_FILE = (
    "/mnt/backup_ETH/Marina/WACCM/CHEM_2000_restart/"
    "BWCN.e122.f19_g16.002/BWCN.e122.f19_g16.002.cam.h3.0001-0023.int.nc"
)

OUT_NC = os.path.join(EHF_ROOT, "EHF_vert_std_anom_15dALL_special_45_75N.nc")

N_PREV = 91  # 前一年 Oct–Dec 天数
YEARS_SPECIAL = np.array([8, 13, 14, 19], dtype=int)


def bootstrap_ci(data, nboot=5000, alpha=0.05, rng=None):
    """
    对给定样本 data 进行 bootstrap，返回均值的 (low, high) 置信区间。
    """
    data = np.asarray(data)
    data = data[~np.isnan(data)]
    n = data.size
    if n < 2:
        return np.nan, np.nan

    if rng is None:
        rng = np.random.default_rng()

    means = np.empty(nboot)
    for i in range(nboot):
        samp = rng.choice(data, size=n, replace=True)
        means[i] = np.mean(samp)

    low = np.percentile(means, 100 * alpha / 2.0)
    high = np.percentile(means, 100 * (1 - alpha / 2.0))
    return low, high


def compute_significance_for_baseline(
    base_anom,   # xarray.DataArray(time, lev) baseline 日异常 (wrt 15d-smoothed daily climatology)
    anom_ref,    # np.ndarray(lev, time_index) 特殊年份 composite 异常 (同一基准)
    doy_base,    # baseline 的 DOY 序列 (time,)
    doy_comp,    # composite 的 DOY 序列 (time_index,)
    alpha=0.05,
    nboot=5000,
):
    """
    对每个 (lev, time_index) 计算显著性：
      - t 检验：baseline 日异常是否显著偏离 0（双侧）；以 ref anomaly 为观测值。
      - bootstrap：baseline 日异常的 mean 的 bootstrap CI 是否包含 ref anomaly。
    返回：
      t_sig(lev, time_index), b_sig(lev, time_index)  布尔数组
    """
    base_vals = base_anom.values  # (time, lev)
    lev_n = base_vals.shape[1]
    t_len = anom_ref.shape[1]

    t_sig = np.zeros((lev_n, t_len), dtype=bool)
    b_sig = np.zeros((lev_n, t_len), dtype=bool)

    rng = np.random.default_rng(2025)

    for ti in range(t_len):
        day = int(doy_comp[ti])
        mask_day = (doy_base == day)
        if not np.any(mask_day):
            continue

        day_samples = base_vals[mask_day, :]  # (n_samples, lev)

        for li in range(lev_n):
            col = day_samples[:, li]
            col = col[~np.isnan(col)]
            if col.size < 2:
                continue

            obs = anom_ref[li, ti]
            # t-test
            se = np.std(col, ddof=1) / np.sqrt(col.size)
            if se == 0:
                continue
            tstat = obs / se
            pval = 2 * (1 - t_dist.cdf(abs(tstat), df=col.size - 1))
            t_sig[li, ti] = (pval < alpha)

            # bootstrap CI
            lo, hi = bootstrap_ci(col, nboot=nboot, alpha=alpha, rng=rng)
            if np.isnan(lo) or np.isnan(hi):
                continue
            b_sig[li, ti] = not (lo <= obs <= hi)

    return t_sig, b_sig


def calc_extr_ehf_45_75N():
    """
    从 EXTR EHF 文件中提取 45–75°N HFLUX(time, lev<=850hPa)，
    并在 time 维度上做 15-day centered running mean。
    返回：
      ehf_extr_smooth (time, lev)，lev 为 hPa，去掉 990hPa。
    """
    print("[BlockG_EHF] Reading EXTR EHF:", EXTR_EHF_FILE)
    ds_extr = xr.open_dataset(EXTR_EHF_FILE)

    hflux = ds_extr["HFLUX"]  # (time, lev, lat)
    lev_vals = ds_extr["lev"].values  # hPa-like

    lev_mask = (lev_vals <= 850.0)
    lev_keep = lev_vals[lev_mask]

    # 45–75N 平均
    hflux_45_75 = hflux.sel(lat=slice(45, 75)).mean("lat")  # (time, lev)

    # 选取需要层，并保证 (time, lev)
    ehf_extr = hflux_45_75.sel(lev=lev_keep).transpose("time", "lev")  # (time, lev)

    # 对 EXTR 时间序列做 15d centered running mean
    ehf_extr_smooth = ehf_extr.rolling(time=15, center=True, min_periods=1).mean()

    ds_extr.close()

    print("[BlockG_EHF] EXTR HFLUX 45–75N shape (smoothed):", ehf_extr_smooth.shape)
    print("[BlockG_EHF] lev_keep (hPa):", lev_keep)

    return ehf_extr_smooth, lev_keep


def calc_eddy_heat_flux_from_INT():
    """
    从 INT 文件中计算 v'T'：
      - 对每个 special year 及其前一年，计算 v'T'(time, plev, lat, lon)，
        并 15d 平滑将在 main 中完成（在 lat 平均和插值之后）。
    """
    print("[BlockG_EHF] Reading INT file:", INT_FILE)
    ds_int = xr.open_dataset(INT_FILE)

    V = ds_int["V"]  # (time, plev, lat, lon)
    T = ds_int["T"]
    plev = ds_int["plev"].values  # Pa

    years_int = ds_int.time.dt.year.values.astype(int)
    all_years_int = np.unique(years_int)
    print("[BlockG_EHF] INT years:", all_years_int)

    vt_by_year = {}  # year -> DataArray(time, plev, lat)

    for y in YEARS_SPECIAL:
        for yy in [y - 1, y]:
            if yy in vt_by_year:
                continue
            if yy not in all_years_int:
                print(f"[BlockG_EHF] ⚠ Year {yy:04d} not found in INT, skip.")
                continue

            print(f"[BlockG_EHF] Computing v'T' for year {yy:04d} ...")
            V_y = V.sel(time=ds_int.time.dt.year == yy)
            T_y = T.sel(time=ds_int.time.dt.year == yy)

            V_zm = V_y.mean("lon")
            T_zm = T_y.mean("lon")
            V_prime = V_y - V_zm
            T_prime = T_y - T_zm

            vt_eddy = (V_prime * T_prime).mean("lon")  # (time, plev, lat)

            vt_by_year[yy] = vt_eddy

    ds_int.close()
    return vt_by_year, plev


def main_blockG_EHF():
    # ===== 1. EXTR baseline（15d 平滑后）=====
    ehf_extr_smooth, lev_keep = calc_extr_ehf_45_75N()  # (time, lev)
    lev_n = lev_keep.size

    time_extr = ehf_extr_smooth["time"]
    years_extr = time_extr.dt.year.values.astype(int)
    doy_extr = time_extr.dt.dayofyear.values.astype(int)
    print("[BlockG_EHF] EXTR years:", np.unique(years_extr))

    n_days = int(time_extr.dt.dayofyear.max())
    print("[BlockG_EHF] Days per year (EXTR):", n_days)

    # ===== 2. 基于 15d-smooth EXTR 的 daily climatology =====
    clim_all = ehf_extr_smooth.groupby("time.dayofyear").mean("time")  # (dayofyear, lev)
    clim_std = ehf_extr_smooth.groupby("time.dayofyear").std("time")   # (dayofyear, lev)

    # baseline anomalies: 每个样本对自己的 DOY 气候态 (平滑版) 求异常
    clim_all_sel = clim_all.sel(dayofyear=time_extr.dt.dayofyear)
    base_anom_all = ehf_extr_smooth - clim_all_sel  # (time, lev)

    # composite DOY: [Oct..Dec, Jan..Sep]
    doy_comp = np.concatenate([
        np.arange(n_days - N_PREV + 1, n_days + 1, dtype=int),
        np.arange(1, n_days - N_PREV + 1, dtype=int),
    ])
    t_len = doy_comp.size
    print("[BlockG_EHF] Composite DOY sequence:", doy_comp[:5], "...", doy_comp[-5:])

    # 从 daily climatology 抽取 composite 的 (lev, t) 背景
    clim_all_comp = clim_all.sel(dayofyear=doy_comp)  # (time_index, lev)
    clim_std_comp = clim_std.sel(dayofyear=doy_comp)  # (time_index, lev)

    clim_all_comp_vals = clim_all_comp.transpose("lev", "dayofyear").values  # (lev, t)
    clim_std_comp_vals = clim_std_comp.transpose("lev", "dayofyear").values  # (lev, t)

    # ===== 3. INT v'T' + lat 平均 + 15d 平滑 + 插值 =====
    vt_by_year, plev_int = calc_eddy_heat_flux_from_INT()
    plev_hpa_int = plev_int / 100.0

    plev_da = xr.DataArray(plev_hpa_int, dims=("plev_hpa",))
    lev_keep_da = xr.DataArray(lev_keep, dims=("lev",))

    n_ref = len(YEARS_SPECIAL)
    std_anom_arr = np.full((n_ref, lev_n, t_len), np.nan, dtype=float)
    t_sig_arr = np.zeros((n_ref, lev_n, t_len), dtype=bool)
    b_sig_arr = np.zeros((n_ref, lev_n, t_len), dtype=bool)

    for yi, y in enumerate(YEARS_SPECIAL):
        print(f"\n[BlockG_EHF] ==== Special year {y:04d} ====")

        if (y not in vt_by_year) or ((y - 1) not in vt_by_year):
            print(f"[BlockG_EHF] ⚠ Year {y:04d} or {y-1:04d} v'T' missing, skip.")
            continue

        vt_cur = vt_by_year[y]      # (time, plev, lat)
        vt_prev = vt_by_year[y - 1] # (time, plev, lat)

        if vt_cur.sizes["time"] < n_days or vt_prev.sizes["time"] < n_days:
            print(f"[BlockG_EHF] ⚠ Year {y:04d} or {y-1:04d} has <{n_days} days, skip.")
            continue

        # 45–75N 平均
        vt_cur_45_75 = vt_cur.sel(lat=slice(45, 75)).mean("lat")   # (time, plev)
        vt_prev_45_75 = vt_prev.sel(lat=slice(45, 75)).mean("lat") # (time, plev)

        # 转成 plev_hpa 维度，便于插值
        vt_cur_lev = vt_cur_45_75.assign_coords(plev_hpa=("plev", plev_hpa_int)) \
                                   .swap_dims({"plev": "plev_hpa"})
        vt_prev_lev = vt_prev_45_75.assign_coords(plev_hpa=("plev", plev_hpa_int)) \
                                    .swap_dims({"plev": "plev_hpa"})

        # 插值到 EXTR 的 lev_keep
        vt_cur_interp = vt_cur_lev.interp(plev_hpa=lev_keep_da)   # (time, lev)
        vt_prev_interp = vt_prev_lev.interp(plev_hpa=lev_keep_da) # (time, lev)

        # 对 v'T' time series 也做 15d centered running mean
        vt_cur_smooth = vt_cur_interp.rolling(time=15, center=True, min_periods=1).mean()
        vt_prev_smooth = vt_prev_interp.rolling(time=15, center=True, min_periods=1).mean()

        vt_cur_vals = vt_cur_smooth.transpose("time", "lev").values   # (time, lev)
        vt_prev_vals = vt_prev_smooth.transpose("time", "lev").values # (time, lev)

        # 构造 composite: [前一年 Oct–Dec, 当年 Jan–Sep]
        ref_comp_vals = np.concatenate([
            vt_prev_vals[n_days - N_PREV:n_days, :],   # (91, lev)
            vt_cur_vals[0:n_days - N_PREV, :],         # (274, lev)
        ], axis=0)  # (365, lev)

        ref_comp = ref_comp_vals.T  # (lev, t_len)

        # === 4. anomaly & std.anom 相对于 15d-smoothed daily climatology ===
        anom_ref = ref_comp - clim_all_comp_vals  # (lev, t_len)

        with np.errstate(divide="ignore", invalid="ignore"):
            std_anom = anom_ref / clim_std_comp_vals
            std_anom[~np.isfinite(std_anom)] = np.nan

        std_anom_arr[yi, :, :] = std_anom

        # === 5. 显著性（基于 smoothed EXTR baseline） ===
        print(f"[BlockG_EHF]  t/boot significance for {y:04d} ...")
        t_sig, b_sig = compute_significance_for_baseline(
            base_anom_all,
            anom_ref,
            doy_extr,
            doy_comp,
            alpha=0.05,
            nboot=5000,
        )
        t_sig_arr[yi, :, :] = t_sig
        b_sig_arr[yi, :, :] = b_sig

    # ===== 6. 写出结果 =====
    ds_out = xr.Dataset(
        coords={
            "ref_year": ("ref_year", YEARS_SPECIAL),
            "lev": ("lev", lev_keep),
            "time_index": ("time_index", np.arange(t_len)),
            "dayofyear": ("time_index", doy_comp),
        },
        data_vars={
            "std_anom": (("ref_year", "lev", "time_index"), std_anom_arr),
            "clim_all_comp": (("lev", "time_index"), clim_all_comp_vals),
            "clim_std_comp": (("lev", "time_index"), clim_std_comp_vals),
            "t_sig": (("ref_year", "lev", "time_index"), t_sig_arr),
            "b_sig": (("ref_year", "lev", "time_index"), b_sig_arr),
        },
    )

    ds_out["std_anom"].attrs["units"] = "σ (relative to 15d-smoothed daily climatology)"
    ds_out["clim_all_comp"].attrs["units"] = "K m s-1"
    ds_out["clim_std_comp"].attrs["units"] = "K m s-1"
    ds_out["lev"].attrs["units"] = "hPa"

    ds_out.attrs["description"] = (
        "Eddy heat flux v'T' (45–75N) standardized anomalies (σ) "
        "for special years relative to EXTR 15-day-smoothed daily climatology. "
        "EXTR baseline and INT v'T' time series are all 15-day centered-smoothed "
        "before forming anomalies. Significance masks from t-test and bootstrap."
    )

    ds_out.to_netcdf(OUT_NC)
    print(f"[BlockG_EHF] Saved dataset to: {OUT_NC}")
    print("[BlockG_EHF] Done.")


if __name__ == "__main__":
    main_blockG_EHF()


In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# ============================================================
# BlockD (T) — 绘制 60–90°N 最低温度时间×高度 anomaly 图
#
# 更新内容 (2025-11-21):
#   1) 去掉原先的 baseline climatology 等值线
#   2) 改为标注 baseline climatology 中 T < 197 K
#      (Cl activation threshold) 的区域：
#        - 半透明冷区填色
#        - 197K 边界虚线等值线
#   3) 原有显著性非显著 hatch 完全保持
# ============================================================

import os
import numpy as np
import xarray as xr
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

T_ROOT = "/home/weiji/test_code/plots/New_weiji_general_why0008important/General_T"
os.makedirs(T_ROOT, exist_ok=True)

IN_NC = os.path.join(T_ROOT, "T_min_vert_anom_special_60_90N.nc")

XTICKS = [0, 31, 61, 91, 122, 150, 181, 211, 242, 273, 304, 334]
XTICK_LABELS = ["Oct", "Nov", "Dec", "Jan", "Feb", "Mar", "Apr",
                "May", "June", "July", "Aug", "Sep"]

# === Cl activation threshold (same as BlockB) ===
CL_THRESH = 197.0  # K

mpl.rc("xtick", direction="out", labelsize=8)
mpl.rc("ytick", direction="out", labelsize=8)
mpl.rcParams["xtick.major.width"] = 0.8
mpl.rcParams["ytick.major.width"] = 0.8
mpl.rc("font", family="sans-serif")
mpl.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "Liberation Sans"]
mpl.rc("axes", titlesize=11, labelsize=9, linewidth=0.8)
mpl.rc("legend", fontsize=7, frameon=False)
mpl.rc("figure", figsize=(6, 4), dpi=300)
mpl.rc("savefig", bbox="tight", pad_inches=0.1)


def guess_p_2_pa(lev_vals):
    lev_vals = np.asarray(lev_vals)
    maxv = np.nanmax(lev_vals)
    if maxv <= 2000.0:
        return lev_vals * 100.0, "hPa"
    else:
        return lev_vals, "Pa"


# === Cl activation threshold (same as BlockB) ===
CL_THRESH = 197.0  # K

def plot_time_height_anom(
    x_idx,
    lev_vals,
    anom_vals,
    clim_vals,
    nonsig_mask,
    ref_year,
    baseline_label,
    test_label,
    outfile,
):
    fig, ax = plt.subplots()

    x = np.asarray(x_idx)
    p_pa, p_unit = guess_p_2_pa(lev_vals)

    # ===== 1) anomaly 底图 =====
    vlim = np.nanmax(np.abs(anom_vals))
    if np.isfinite(vlim) and vlim > 0:
        vmax = min(vlim, 20.0)
    else:
        vmax = 10.0
    levels = np.linspace(-vmax, vmax, 31)

    cf = ax.contourf(
        x, p_pa, anom_vals,
        levels=levels,
        cmap="RdBu_r",
        extend="both",
        antialiased=True,
        alpha=0.85,
        zorder=1,
    )

    # ===== 2) 197K 等值线：背景clim(虚线紫) vs 当年T(实线紫) =====
    # 背景 climatology 197K 虚线
    ax.contour(
        x, p_pa, clim_vals,
        levels=[CL_THRESH],
        colors="purple",
        linewidths=1.2,
        linestyles="--",
        zorder=3,
    )

    # 该年份实际温度 = anomaly + baseline climatology
    ref_temp_vals = anom_vals + clim_vals

    # 当年 197K 实线
    ax.contour(
        x, p_pa, ref_temp_vals,
        levels=[CL_THRESH],
        colors="purple",
        linewidths=1.5,
        linestyles="-",
        zorder=4,
    )

    # ===== 3) 非显著 hatch（保持原逻辑） =====
    legend_handles = []

    if nonsig_mask is not None:
        mask_int = nonsig_mask.astype(int)
        ax.contourf(
            x, p_pa, mask_int,
            levels=[0.5, 1.5],
            colors="none",
            hatches=["///"],
            alpha=0,
            zorder=5,
        )
        legend_handles.append(
            Patch(
                facecolor="white",
                hatch="///",
                edgecolor="black",
                label="Not significant (p ≥ 0.05)",
            )
        )

    # legend：两条 197K 线
    legend_handles.append(
        mpl.lines.Line2D([0], [0], color="purple", lw=1.2, ls="--",
                         label=f"Baseline clim T = {CL_THRESH:.0f} K")
    )
    legend_handles.append(
        mpl.lines.Line2D([0], [0], color="purple", lw=1.5, ls="-",
                         label=f"Year {int(ref_year):04d} T = {CL_THRESH:.0f} K")
    )

    ax.legend(handles=legend_handles, loc="upper right")

    # ===== 坐标轴与标题 =====
    ax.set_yscale("log")
    ax.invert_yaxis()
    ax.set_ylabel(f"Pressure ({p_unit})")

    ax.set_xlim(x[0], x[-1])
    ax.set_xticks(XTICKS)
    ax.set_xticklabels(XTICK_LABELS)

    ax.grid(True, which="major", linestyle="--", linewidth=0.4, alpha=0.5)

    ax.set_title(
        f"Min T anomaly (K), 60–90°N, Year {int(ref_year):04d}\n"
        f"Baseline: {baseline_label}, Mask: non-significant by {test_label}"
    )

    cbar = plt.colorbar(cf, ax=ax, pad=0.02)
    cbar.set_label("Min T anomaly (K)")

    plt.savefig(outfile, dpi=300)
    plt.close(fig)
    print("  Saved:", outfile)



def main_blockD_T():
    ds = xr.open_dataset(IN_NC)

    ref_years = ds["ref_year"].values
    lev = ds["lev"].values
    time_index = ds["time_index"].values

    anom_all = ds["anom_all"].values
    anom_low = ds["anom_low25"].values
    clim_all = ds["clim_all_comp"].values
    clim_low = ds["clim_low25_comp"].values

    t_sig_all = ds["t_sig_all"].values
    b_sig_all = ds["b_sig_all"].values
    t_sig_low = ds["t_sig_low25"].values
    b_sig_low = ds["b_sig_low25"].values

    ds.close()

    for yi, y in enumerate(ref_years):
        # baseline all-years, t-test
        nonsig_t_all = ~t_sig_all[yi, :, :]
        outfile = os.path.join(
            T_ROOT,
            f"Tmin_anom_60_90N_year{int(y):04d}_vsALL_t_nonsig.png"
        )
        plot_time_height_anom(
            time_index,
            lev,
            anom_all[yi, :, :],
            clim_all,
            nonsig_t_all,
            ref_year=y,
            baseline_label="EXTR all-years climatology",
            test_label="t-test",
            outfile=outfile,
        )

        # baseline all-years, bootstrap
        nonsig_b_all = ~b_sig_all[yi, :, :]
        outfile = os.path.join(
            T_ROOT,
            f"Tmin_anom_60_90N_year{int(y):04d}_vsALL_bootstrap_nonsig.png"
        )
        plot_time_height_anom(
            time_index,
            lev,
            anom_all[yi, :, :],
            clim_all,
            nonsig_b_all,
            ref_year=y,
            baseline_label="EXTR all-years climatology",
            test_label="bootstrap",
            outfile=outfile,
        )

        # baseline low-25, t-test
        nonsig_t_low = ~t_sig_low[yi, :, :]
        outfile = os.path.join(
            T_ROOT,
            f"Tmin_anom_60_90N_year{int(y):04d}_vsLOW25_t_nonsig.png"
        )
        plot_time_height_anom(
            time_index,
            lev,
            anom_low[yi, :, :],
            clim_low,
            nonsig_t_low,
            ref_year=y,
            baseline_label="EXTR low-25% climatology",
            test_label="t-test",
            outfile=outfile,
        )

        # baseline low-25, bootstrap
        nonsig_b_low = ~b_sig_low[yi, :, :]
        outfile = os.path.join(
            T_ROOT,
            f"Tmin_anom_60_90N_year{int(y):04d}_vsLOW25_bootstrap_nonsig.png"
        )
        plot_time_height_anom(
            time_index,
            lev,
            anom_low[yi, :, :],
            clim_low,
            nonsig_b_low,
            ref_year=y,
            baseline_label="EXTR low-25% climatology",
            test_label="bootstrap",
            outfile=outfile,
        )

    print("[BlockD_T] All figures generated.")


if __name__ == "__main__":
    main_blockD_T()
